## Proyecto final


In [1]:
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit


I. Descripción del problema

El dengue es una enfermedad viral transmitida por el mosquito Aedes Aegypti. En Colombia, su comportamiento epidemiológico presenta variaciones por temporalidad, ciclos, clima y estacionalidad. El objetivo de este trabajo es utilizar datos históricos oficiales de dengue correspondientes a los años 2022, 2023 y 2024, y entrenar modelos supervisados de Machine Learning que permitan proyectar cuál sería el comportamiento del dengue en la siguiente ventana de tiempo futura.

Este es un problema de regresión supervisada porque deseamos estimar una variable numérica (cantidad de casos). El análisis se realizará a nivel nacional, sumando casos semanales para construir una serie temporal agregada.

El objetivo principal es comparar el rendimiento de distintos modelos supervisados:

- Regresión Lineal Multivariada

- Árbol de decisión

- Random Forest

- Redes neuronales (MLP/DNN)

al momento de predecir valores futuros de casos de dengue en Colombia.

II. Inspección del Dataset inicial

El dataset original proviene de datos de vigilancia del sistema colombiano (años 2022-2024). Contiene múltiples columnas clínicas, administrativas, geográficas y epidemiológicas. Durante la inspección se identificó que muchas variables no son necesarias para el objetivo predicitivo planteado, por lo tanto, se realizará un proceso de limpieza y selección de variables.

In [ ]:
df = pd.read_excel("Datos_22-24.xlsx", sheet_name="22-24", engine="openpyxl")

print("Filas y columnas:", df.shape)
df.head()

III. Preprocesamiento del conjunto de datos

Estrategias utilizadas para preparar los datos

Durante la inspección inicial del conjunto de datos se identificó que el archivo original contenía una gran cantidad de variables relacionadas con clínica, ubicación geográfica, tipo de caso, entidad de aseguramiento, clasificación y variables textuales que no aportaban directamente al objetivo de predicción en este estudio. Debido a que este trabajo busca predecir el comportamiento futuro del dengue a nivel nacional, se decidió realizar una reducción de características, conservando únicamente las columnas relevantes para construir una serie temporal: la fecha de notificación (FEC_NOT) y el número de casos confirmados (confirmados).

Posteriormente se aplicaron las siguientes estrategias de preparación:

- Conversión de variables: la columna FEC_NOT fue convertida a tipo datetime para poder manipular adecuadamente la información temporal.

- Manejo de valores faltantes: se eliminaron registros que tenían fecha o número de casos faltantes. En la agregación semanal, valores faltantes resultantes se rellenaron con 0 para mantener la continuidad temporal.

- Agregación temporal: los registros individuales fueron transformados a un formato semanal sumando el número de casos confirmados por semana para obtener una serie temporal nacional.

- Reducción dimensional: se eliminaron todas las columnas restantes que no aportaban al objetivo del modelo, reduciendo así ruido y complejidad.

- Ingeniería de características: se generaron variables derivadas de series temporales (lag features y medias móviles) y variables estacionales (semana del año, mes, año) con el fin de transformar el problema de series temporales en un problema de regresión supervisada.

- Separación temporal de datos: la partición entre entrenamiento y prueba se realizó respetando el orden temporal de la serie (80% entrenamiento, 20% prueba), evitando introducir fuga de información.

- Estandarización: durante el entrenamiento, los modelos se ejecutan dentro de pipelines que realizan normalización de variables numéricas, lo cual es importante especialmente en redes neuronales y modelos como Random Forest.

Estas decisiones de preprocesamiento están directamente soportadas en la inspección y análisis del dataset, y permiten adaptar el conjunto de datos al tipo de tarea requerida por los modelos supervisados propuestos en este trabajo.

In [ ]:
# Selección de columnas relevantes
df = df[['FEC_NOT','confirmados']]

# Conversión de fecha
df['FEC_NOT'] = pd.to_datetime(df['FEC_NOT'])

# Eliminación filas con fecha o casos faltantes
df = df.dropna(subset=['FEC_NOT','confirmados'])

df.head()


IV. Construcción de la serie nacional semanal

Se agregan los casos confirmados semanalmente y se construyen variables de series temporales como lags y medias móviles, para convertir el problema de predicción de series de tiempo en un problema supervisado estándar.

In [4]:
# crear semana
df['year_week'] = df['FEC_NOT'].dt.to_period('W').apply(lambda r: r.start_time)

series_weekly = df.groupby('year_week')['confirmados'].sum().sort_index()
series_weekly = series_weekly.asfreq('W-MON').fillna(0)
series_weekly = series_weekly.rename("cases")

def make_features(series, lags=[1,2,3,4], windows=[3,6]):
    df_feat = pd.DataFrame({'cases': series})
    
    # Crear variables de rezago (lags)
    for lag in lags:
        df_feat[f'lag_{lag}'] = df_feat['cases'].shift(lag)
    
    # Crear medias móviles
    for w in windows:
        df_feat[f'roll_mean_{w}'] = df_feat['cases'].shift(1).rolling(window=w, min_periods=1).mean()
    
    # Extraer características temporales usando isocalendar()
    df_feat['weekofyear'] = df_feat.index.isocalendar().week
    df_feat['month'] = df_feat.index.month
    df_feat['year'] = df_feat.index.year
    
    return df_feat.dropna()

V. Entrenamiento, ajuste de hiperparámetros y evaluación

Se entrenan los 4 modelos exigidos usando GridSearchCV y TimeSeriesSplit debido a que se trata de series temporales donde no se debe mezclar aleatoriamente el tiempo.

Métricas de evaluación:

- MAE

- MSE

- R²

In [ ]:
# Crear features para el modelo
df_feat = make_features(series_weekly)

# Verificar que se creó correctamente
print("Shape del DataFrame con features:", df_feat.shape)
print("\nPrimeras filas:")
print(df_feat.head())

# split temporal (no shuffle)
split_idx = int(len(df_feat)*0.8)
X = df_feat.drop(columns=['cases'])
y = df_feat['cases']
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

tscv = TimeSeriesSplit(n_splits=5)

# Modelos
models = {
    'linear': Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())]),
    'tree': Pipeline([('scaler', StandardScaler()), ('tree', DecisionTreeRegressor(random_state=42))]),
    'rf': Pipeline([('scaler', StandardScaler()), ('rf', RandomForestRegressor(random_state=42))]),
    'mlp': Pipeline([('scaler', StandardScaler()), ('mlp', MLPRegressor(max_iter=1000, random_state=42))]),
}

# Grids
param_grids = {
    'linear': {'lr__fit_intercept':[True, False]},
    'tree': {'tree__max_depth':[3,5,10,None], 'tree__min_samples_split':[2,5,10]},
    'rf': {'rf__n_estimators':[50,100,200], 'rf__max_depth':[5,10,None]},
    'mlp': {'mlp__hidden_layer_sizes':[(50,),(100,),(100,50)], 'mlp__alpha':[0.0001,0.001]}
}

# Entrenamiento + evaluación
results = {}
for name, pipe in models.items():
    print("Entrenando:", name)
    grid = GridSearchCV(pipe, param_grids[name], cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    best = grid.best_estimator_
    y_pred = best.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'best_params': grid.best_params_, 'mae': mae, 'mse': mse, 'r2': r2, 'pred': y_pred}
    print(name, results[name])

# Dataframe comparativo
res_df = pd.DataFrame({k: {'MAE': v['mae'], 'MSE': v['mse'], 'R2': v['r2']} for k,v in results.items()}).T
res_df